# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
# imports
from os import getenv
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import gradio as gr # oh yeah!

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:1b'

# set up environment
load_dotenv(override=True)
api_key = getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

system_prompt = """
You are a medical doctor (MD) who is an expert in clinical chemistry.
"""

def demystify(message, model):
    print(f"demystify has been called with message={message}, model={model}")
    llm = openai if model == "GPT" else ollama
    stream = llm.chat.completions.create(
        model=MODEL_GPT if model == "GPT" else MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": message}
        ],
        stream=True
    )
    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

message_input = gr.Textbox(label="Your message:", info="Enter a techncial message for the LLM", lines=7)
model_selector = gr.Dropdown(["GPT", "Ollama"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=demystify,
    title="Chat GPT in Medical Doctor Mode",
    inputs=[message_input, model_selector],
    outputs=[message_output],
    flagging_mode="never"
)

view.launch()
